# 03 · Scope (LEGB), Lifetime, `global`, `nonlocal` & Closures

**Module 05 – Functions** · Estimated time: 45 min

### 🎯 What you will learn
- What **scope** and **lifetime** of a variable mean
- The **LEGB** lookup rule: Local → Enclosing → Global → Built-in
- The `global` and `nonlocal` keywords (and why to avoid them)
- The `UnboundLocalError` trap
- Nested functions and **closures** (the basis of decorators)
- `globals()` and `locals()`

## 1. The question (📗)

In [1]:
x = 25

def printer():
    x = 50
    return x

print(x)
print(printer())
print(x)          # still 25!

25
50
25


**Scope** = the region of code where a name is visible. Assigning `x = 50` inside `printer` creates a **new local name** that only exists inside the function. The global `x` is untouched.

**Lifetime** = how long the variable exists. Local variables are created when the function is called and destroyed when it returns.

## 2. The LEGB rule

When Python looks up a name, it searches these scopes **in order** and uses the first match:

```
┌─ B: Built-in ── names preassigned in Python (len, print, range, ...) ─┐
│ ┌─ G: Global ── names at the top level of the module/file ─────────┐  │
│ │ ┌─ E: Enclosing ── locals of any enclosing function(s) ───────┐  │  │
│ │ │ ┌─ L: Local ── names assigned inside the current function ┐ │  │  │
│ │ │ └─────────────────────────────────────────────────────────┘ │  │  │
│ │ └─────────────────────────────────────────────────────────────┘  │  │
│ └──────────────────────────────────────────────────────────────────┘  │
└───────────────────────────────────────────────────────────────────────┘
```

### Quick examples of LEGB (📗)

In [2]:
# L: Local
f = lambda x: x ** 2          # x is local to the lambda
print(f(4))

# E: Enclosing function locals
name = "This is a global name"

def greet():
    name = "Sammy"                 # enclosing for hello()
    def hello():
        print("Hello " + name)     # not local → found in the enclosing scope
    hello()

greet()

# G: Global
print(name)

# B: Built-in
print(len)

16
Hello Sammy
This is a global name
<built-in function len>


## 3. Local variables (📘 handbook)

In [3]:
x = 10              # global variable

def my_func():
    x = 5           # local variable (a different x)
    print(x)

my_func()           # 5
print(x)            # 10 (global x remains unchanged)

5
10


In [4]:
def make_secret():
    secret = 42

make_secret()
print(secret)       # ❌ NameError: the local died when the function returned

NameError: name 'secret' is not defined

## 4. The `global` statement

To **assign** to a global variable from inside a function, declare it `global`:

In [5]:
x = 50

def func():
    global x
    print("This function is now using the global x!")
    print("Because of the global statement, x is:", x)
    x = 2
    print("Ran func(), changed global x to", x)

print("Before calling func(), x is:", x)
func()
print("Value of x (outside of func()) is:", x)

Before calling func(), x is: 50
This function is now using the global x!
Because of the global statement, x is: 50
Ran func(), changed global x to 2
Value of x (outside of func()) is: 2


> ⚠️ **Avoid `global` in real code.** It makes functions depend on hidden state, which makes bugs hard to track. Prefer **passing values in and returning results out**:

In [6]:
x = 50

def func(x):
    x = 2
    return x

x = func(x)       # explicit and clear
print(x)

2


## 5. ❌ The `UnboundLocalError` trap

If a function **assigns** to a name *anywhere* in its body, that name is local **everywhere** in the function, even before the assignment:

In [7]:
counter = 0

def increment():
    counter += 1        # reads counter as a LOCAL before it has a value
    return counter

increment()

UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value

Fix: pass it in and return it (best), or declare `global counter`.

## 6. `nonlocal`: modifying an enclosing variable

In [8]:
def outer():
    count = 0
    def inner():
        nonlocal count      # refers to outer's count
        count += 1
        return count
    inner(); inner()
    return count

print(outer())

2


## 7. Closures

A **closure** is an inner function that **remembers** variables from its enclosing scope, *even after the outer function has finished*.

In [9]:
def make_multiplier(factor):
    def multiply(number):
        return number * factor     # 'factor' is remembered
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10), triple(10))
print(double.__closure__[0].cell_contents)    # peek at the remembered value

20 30
2


In [10]:
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c1 = make_counter()
c2 = make_counter()
print(c1(), c1(), c1(), c2())     # each closure has its own state

1 2 3 1


Closures power **decorators** (Module 09), callbacks and factories.

### ❌ Late-binding gotcha in loops

In [11]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])        # [2, 2, 2]: all see the final i

funcs = [lambda i=i: i for i in range(3)]   # bind the current value as a default
print([f() for f in funcs])        # [0, 1, 2]

[2, 2, 2]
[0, 1, 2]


## 8. Inspecting scopes: `globals()` and `locals()` (📗 Decorators lesson)

In [12]:
s = "Global Variable"

def check_for_locals():
    local_var = 1
    print(locals())

check_for_locals()
print("s" in globals(), globals()["s"])

{'local_var': 1}
True Global Variable


## 9. Built-ins can be shadowed (but shouldn't be)

In [13]:
def demo():
    len = lambda x: "shadowed!"     # local name hides the built-in len
    return len([1, 2, 3])

print(demo(), len([1, 2, 3]))

shadowed! 3


## 📝 Summary
- Scope = where a name is visible; lifetime = how long it exists.
- Lookup order: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.
- Assigning inside a function creates a local name, unless declared `global`/`nonlocal`.
- Prefer parameters & return values over `global`.
- Closures remember enclosing variables; they're the basis of decorators.

## 🏋️ Practice
1. Predict the output:
   ```python
   x = "global"
   def outer():
       x = "enclosing"
       def inner():
           print(x)
       inner()
   outer()
   ```
2. Write `make_power(n)` returning a function that raises its argument to `n`.
3. Write `make_accumulator()` returning a function that adds its argument to a running total and returns the total.
4. Why does this fail, and how do you fix it without `global`?
   ```python
   total = 0
   def add(n):
       total = total + n
   ```

<details><summary>Show solutions</summary>

```python
# 1 → enclosing

def make_power(n):
    return lambda x: x ** n

def make_accumulator():
    total = 0
    def add(n):
        nonlocal total
        total += n
        return total
    return add

# 4: total is local (assigned in the function) → UnboundLocalError.
total = 0
def add(total, n):
    return total + n
total = add(total, 5)
print(total)
```
</details>